# Entrenamiento visual Full384 — implementación reproducible

Este notebook reconstruye de forma limpia la configuración final del clasificador visual **ResNet18 Full384** a partir del código y de los artefactos conservados del experimento.

> **Importante:** no es el notebook histórico original ejecutado en Google Colab. Es una implementación reproducible de la configuración final documentada.

La evaluación final se realizó con particionado **a nivel de evento** para evitar fuga de información entre frames temporalmente próximos del mismo incendio.


## Configuración experimental

- Backbone: ResNet18 con pesos ImageNet.
- Entrada: 384×384.
- Optimizador: AdamW.
- Learning rate: `3e-4`.
- Weight decay: `1e-4`.
- Scheduler: cosine.
- Batch físico: 16.
- Acumulación de gradiente: 4.
- Batch efectivo: 64.
- AMP activado.
- Evaluación Full384: `Resize(440) -> CenterCrop(384)`.


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import pandas as pd
import torch
import torch.nn as nn
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms

from src.visual_model import build_full384_model

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]


## Manifiestos de entrenamiento y validación

Los CSV deben contener las columnas:

```text
path,label,event_id
```

El split debe prepararse previamente **por evento**, no por imagen.


In [ ]:
TRAIN_CSV = ROOT / "data" / "train_manifest.csv"
VAL_CSV = ROOT / "data" / "val_manifest.csv"

class ManifestDataset(Dataset):
    def __init__(self, csv_path, train=True):
        self.df = pd.read_csv(csv_path)
        if train:
            self.transform = transforms.Compose([
                transforms.Resize(440),
                transforms.RandomResizedCrop(384, scale=(0.65, 1.0)),
                transforms.RandomHorizontalFlip(),
                transforms.ColorJitter(
                    brightness=0.2,
                    contrast=0.2,
                    saturation=0.2,
                    hue=0.05,
                ),
                transforms.ToTensor(),
                transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
            ])
        else:
            self.transform = transforms.Compose([
                transforms.Resize(440),
                transforms.CenterCrop(384),
                transforms.ToTensor(),
                transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
            ])

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(row["path"]).convert("RGB")
        x = self.transform(image)
        y = torch.tensor(float(row["label"]), dtype=torch.float32)
        return x, y


In [ ]:
BATCH_SIZE = 16
ACCUMULATION = 4
EPOCHS = 30

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = build_full384_model(pretrained=True).to(device)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=3e-4,
    weight_decay=1e-4,
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=EPOCHS,
)
loss_fn = nn.BCEWithLogitsLoss()
scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())


## Bucle de entrenamiento

La siguiente celda reproduce el núcleo de entrenamiento conservado en `training/train_visual_reference.py`.


In [ ]:
def train_model(model, train_loader, val_loader, output_path):
    best_val = float("inf")
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    for epoch in range(1, EPOCHS + 1):
        model.train()
        optimizer.zero_grad(set_to_none=True)

        for step, (x, y) in enumerate(train_loader, start=1):
            x, y = x.to(device), y.to(device)

            with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
                logits = model(x).squeeze(-1)
                loss = loss_fn(logits, y) / ACCUMULATION

            scaler.scale(loss).backward()

            if step % ACCUMULATION == 0 or step == len(train_loader):
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)

        scheduler.step()

        model.eval()
        total_loss, n = 0.0, 0
        with torch.no_grad():
            for x, y in val_loader:
                x, y = x.to(device), y.to(device)
                logits = model(x).squeeze(-1)
                loss = loss_fn(logits, y)
                total_loss += loss.item() * len(y)
                n += len(y)

        val_loss = total_loss / max(n, 1)
        print(f"epoch={epoch:03d} val_loss={val_loss:.6f}")

        if val_loss < best_val:
            best_val = val_loss
            torch.save(
                {
                    "epoch": epoch,
                    "model_state_dict": model.state_dict(),
                    "val_loss": val_loss,
                    "config": {
                        "input": 384,
                        "backbone": "resnet18",
                        "lr": 3e-4,
                        "weight_decay": 1e-4,
                        "batch_size": BATCH_SIZE,
                        "gradient_accumulation": ACCUMULATION,
                    },
                },
                output_path,
            )


## Resultados finales conservados

El experimento Full384 se evaluó sobre **3 761 imágenes de test correspondientes a 50 eventos**, usando umbral 0.50.

| Métrica | Full384 |
|---|---:|
| Accuracy | 0.78756 |
| Balanced Accuracy | 0.79894 |
| Precision | 0.90195 |
| Recall | 0.69057 |
| F1 | 0.78223 |
| AUC | 0.83860 |
| FPR | 0.09269 |
| TN / FP / FN / TP | 1527 / 156 / 643 / 1435 |

En la comparación controlada con Full224, los falsos positivos pasaron de **301 a 156**.


## Trazabilidad

Los valores anteriores proceden de los artefactos conservados del experimento final (`metrics_resnet18_384_controlled_recovered.csv` y comparación controlada 224/384). La documentación metodológica completa está en `docs/reproducibility.md`.
